# LEVEL 3 -> TASK 2

In [55]:
# Task 2: Support Vector Machine (SVM)
# Dataset: Sentiment dataset.csv

In [56]:
# IMPORTING LIBRARIES
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer

In [57]:
# 1. Load Dataset
df = pd.read_csv("Sentiment dataset.csv")

print("First 5 rows:")
print(df.head())

First 5 rows:
   Unnamed: 0.1  Unnamed: 0  \
0             0           0   
1             1           1   
2             2           2   
3             3           3   
4             4           4   

                                                Text    Sentiment  \
0   Enjoying a beautiful day at the park!        ...   Positive     
1   Traffic was terrible this morning.           ...   Negative     
2   Just finished an amazing workout! 💪          ...   Positive     
3   Excited about the upcoming weekend getaway!  ...   Positive     
4   Trying out a new recipe for dinner tonight.  ...   Neutral      

             Timestamp            User     Platform  \
0  2023-01-15 12:30:00   User123          Twitter     
1  2023-01-15 08:45:00   CommuterX        Twitter     
2  2023-01-15 15:45:00   FitnessFan      Instagram    
3  2023-01-15 18:20:00   AdventureX       Facebook    
4  2023-01-15 19:55:00   ChefCook        Instagram    

                                     Hashtags  Retwee

In [58]:
# 2. Handle Missing Data
df.dropna(inplace=True)

In [59]:
# 3. Encode Target
le = LabelEncoder()
df['Sentiment'] = le.fit_transform(df['Sentiment'])

In [60]:
# 4. Remove rare classes (very important)
counts = df['Sentiment'].value_counts()
valid_classes = counts[counts > 1].index
df = df[df['Sentiment'].isin(valid_classes)]

In [61]:
# 5. Features & Target
X = df['Text']
y = df['Sentiment']

In [62]:
# 6. TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_vect = vectorizer.fit_transform(X)

In [63]:
# 7. Train-Test Split (NO stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X_vect, y, test_size=0.2, random_state=42
)

In [64]:
# ================= MODEL 1 =================
model_linear = SVC(kernel='linear', class_weight='balanced', probability=True)
model_linear.fit(X_train, y_train)

y_pred_linear = model_linear.predict(X_test)
y_prob_linear = model_linear.predict_proba(X_test)

In [65]:
# ================= MODEL 2 =================
model_rbf = SVC(kernel='rbf', class_weight='balanced', probability=True)
model_rbf.fit(X_train, y_train)

y_pred_rbf = model_rbf.predict(X_test)
y_prob_rbf = model_rbf.predict_proba(X_test)

In [66]:
# ================= SAFE EVALUATION =================
def evaluate(y_test, y_pred, y_prob, name):
    print(f"\n{name} Results:")

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

    # -------- SAFE AUC --------
    try:
        if len(set(y_test)) == 2:
            auc = roc_auc_score(y_test, y_prob[:, 1])
        else:
            auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
        print("AUC      :", auc)
    except:
        print("AUC      : Could not be calculated (class mismatch)")

# 8. Evaluate
evaluate(y_test, y_pred_linear, y_prob_linear, "Linear Kernel")
evaluate(y_test, y_pred_rbf, y_prob_rbf, "RBF Kernel")


Linear Kernel Results:
Accuracy : 0.1694915254237288
Precision: 0.19924776233098457
Recall   : 0.1694915254237288
F1 Score : 0.1696798493408663
AUC      : Could not be calculated (class mismatch)

RBF Kernel Results:
Accuracy : 0.11864406779661017
Precision: 0.14414850686037126
Recall   : 0.11864406779661017
F1 Score : 0.11823899371069183
AUC      : Could not be calculated (class mismatch)
